# Tutorial 1: JAX and Flax
**Week 1, Day 1: Basics and JAX**

**By Neuromatch Academy**

__Content creators:__ Adapted for JAX/Flax

__Content reviewers:__

__Production editors:__

---
# Tutorial Objectives

In this tutorial you will learn:
- The basics of **JAX** and **JAX NumPy (`jax.numpy`)** as a replacement for PyTorch tensors
- How JAX handles **randomness** with explicit PRNG keys
- How to manipulate arrays (indexing, reshaping, transposing)
- How JAX manages **devices** (CPU/GPU/TPU) transparently
- How to load datasets with torchvision and convert to JAX arrays
- How to define a neural network with **Flax (`flax.linen`)**
- How to train a network using **Optax** optimizers and **`jax.grad`**
- What **`jax.jit`** is and why it matters

---
# Setup

Throughout your Neuromatch tutorials, most (probably all!) notebooks contain setup cells at the beginning.
**Run these cells first** — they install dependencies, import libraries, and define helper functions.

In [ ]:
# @title Install dependencies
!pip install pandas --quiet

: 

In [ ]:
# @title Install and import feedback gadget
!pip3 install vibecheck datatops --quiet

from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "neuromatch_dl",
            "user_key": "f379rz8y",
        },
    ).render()

feedback_prefix = "W1D1_T1"

In [ ]:
# Imports
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# JAX libraries
import jax
import jax.numpy as jnp
from jax import random as jrandom

# Flax for neural networks
import flax.linen as nn

# Optax for optimizers
import optax

# Torchvision for datasets (we keep these; convert batches to JAX arrays)
from torchvision import datasets
from torchvision.transforms import ToTensor, Compose, Grayscale
from torch.utils.data import DataLoader

In [ ]:
# @title Figure Settings
import logging
logging.getLogger('matplotlib.font_manager').disabled = True

import ipywidgets as widgets
%config InlineBackend.figure_format = 'retina'
plt.style.use("https://raw.githubusercontent.com/NeuromatchAcademy/content-creation/main/nma.mplstyle")

In [ ]:
# @title Helper Functions

def make_key(seed=0):
  """
  Create a JAX PRNG key.

  Args:
    seed: int
      Seed for the random key.

  Returns:
    key: jax.random.PRNGKey
  """
  return jrandom.PRNGKey(seed)


def split_key(key):
  """
  Split a JAX key into two: one to keep, one to use.

  Args:
    key: jax.random.PRNGKey

  Returns:
    key: jax.random.PRNGKey  (keep for next use)
    subkey: jax.random.PRNGKey  (use now)
  """
  return jrandom.split(key)


def checkExercise1(A, B, C, D):
  """
  Helper function for checking Exercise 1.

  Args:
    A: jnp.ndarray of shape (20, 21) consisting of ones.
    B: jnp.ndarray of shape (3, 4)
    C: jnp.ndarray of shape (20, 21)
    D: jnp.ndarray of shape (19,)
  """
  assert jnp.array_equal(A.astype(int), jnp.ones((20, 21), dtype=int)),       f"Got shape {A.shape}, expected (20, 21) of ones"
  assert jnp.array_equal(B, jnp.array(np.vander([1, 2, 3], 4))),       f"B does not match expected Vandermonde matrix"
  assert C.shape == (20, 21), f"C shape wrong: {C.shape}"
  assert D.shape == (19,), f"D shape wrong: {D.shape}"
  print("All correct!")

---
# Section 1: Welcome to Neuromatch Deep Learning Course

*Time estimate: ~25 mins*

In [ ]:
# @title Video 1: Welcome and History
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

This will be an intensive 3 week adventure. We will all learn Deep Learning (DL) together!

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Welcome_and_History_Video")

In [ ]:
# @title Video 2: Why DL is cool
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

**Discuss with your pod: What do you hope to get out of this course? [in about 10 mins]**

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Why_DL_is_cool_Video")

---
# Section 2: The Basics of JAX

*Time estimate: ~2 hours 05 mins*

JAX is a Python library from Google that combines NumPy-style array operations with:
- **Automatic differentiation** via `jax.grad`
- **JIT compilation** via `jax.jit` (runs on CPU, GPU, or TPU)
- **Vectorization** via `jax.vmap`

The core array type is `jax.numpy.ndarray` (also called `jaxlib.xla_extension.ArrayImpl`).
It behaves almost identically to a NumPy array, but computations can be accelerated and differentiated.

## Section 2.1: Creating Arrays

In [ ]:
# @title Video 3: Making Tensors
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Making_Tensors_Video")

There are various ways of creating JAX arrays. The API closely mirrors NumPy.

**Construct arrays directly:**

In [ ]:
# jnp.array() from Python lists, tuples, or numpy arrays

# array from a list
a = jnp.array([0, 1, 2])

# array from a tuple of tuples
b = ((1.0, 1.1), (1.2, 1.3))
b = jnp.array(b)

# array from a numpy array
c = np.ones([2, 3])
c = jnp.array(c)

print(f"Array a: {a}")
print(f"Array b: {b}")
print(f"Array c: {c}")

**Some common array constructors:**

In [ ]:
# The numerical arguments determine the shape of the output array

x = jnp.ones((5, 3))
y = jnp.zeros(2)
z = jnp.empty((1, 1, 5))  # uninitialized values — avoid using these values directly!
print(f"Array x: {x}")
print(f"Array y: {y}")
print(f"Array z: {z}")

> **Note:** Unlike PyTorch's `torch.empty`, `jnp.empty` is rarely used in practice
> because JAX arrays are immutable — you cannot fill them after creation.
> Use `jnp.zeros` or `jnp.full` instead.

**Creating random arrays**

> ⚠️ **Key JAX difference: Explicit PRNG keys**
>
> JAX does *not* have a global random state. Every random operation requires an explicit **key**.
> You split a key into subkeys — one to keep for the next call, one to use now.
> This makes randomness fully reproducible and composable.

In [ ]:
# Create a root key
key = jrandom.PRNGKey(0)

# Split into two: keep `key` for later, use `subkey` now
key, subkey = jrandom.split(key)

# Uniform distribution  (equivalent to torch.rand)
a = jrandom.uniform(subkey, shape=(1, 3))

# Normal distribution  (equivalent to torch.randn)
key, subkey = jrandom.split(key)
b = jrandom.normal(subkey, shape=(3, 4))

# zeros_like and uniform_like
c = jnp.zeros_like(a)
key, subkey = jrandom.split(key)
d = jrandom.uniform(subkey, shape=a.shape)

print(f"Array a: {a}")
print(f"Array b: {b}")
print(f"Array c: {c}")
print(f"Array d: {d}")

*Reproducibility:*
Since JAX requires explicit keys, reproducibility is built-in.
The same key always produces the same result — no global seed needed.

In [ ]:
def simplefun(seed=0):
  """
  Demonstrate reproducibility with JAX keys.

  Args:
    seed: int — the root seed

  Returns:
    Nothing.
  """
  key = jrandom.PRNGKey(seed)
  key, subkey = jrandom.split(key)
  a = jrandom.uniform(subkey, (1, 3))
  key, subkey = jrandom.split(key)
  b = jrandom.normal(subkey, (3, 4))
  print("Array a:", a)
  print("Array b:", b)

simplefun(seed=0)  # Change seed to get different numbers — same seed always gives same result

**Numpy-like number ranges:**

In [ ]:
a = jnp.arange(0, 10, step=1)    # equivalent to torch.arange
b = np.arange(0, 10, step=1)

c = jnp.linspace(0, 5, num=11)   # equivalent to torch.linspace
d = np.linspace(0, 5, num=11)

print(f"JAX array a: {a}")
print(f"NumPy array b: {b}")
print(f"JAX array c: {c}")
print(f"NumPy array d: {d}")

### Coding Exercise 2.1: Creating Arrays

In [ ]:
def array_creation(Z):
  """
  A function that creates various arrays.

  Args:
    Z: numpy.ndarray of shape (3, 4)

  Returns:
    A: jnp.ndarray — shape (20, 21) of ones
    B: jnp.ndarray — values equal to Z
    C: jnp.ndarray — same shape as A, values ~ Uniform(0, 1)
    D: jnp.ndarray — 1D array of even numbers from 4 to 40 inclusive
  """
  #################################################
  ## TODO for students: fill in the missing code
  raise NotImplementedError("Student exercise: fill in the missing code")
  #################################################

  A = ...
  B = ...
  C = ...
  D = ...
  return A, B, C, D

In [ ]:
# to_remove solution
def array_creation(Z):
  """
  A function that creates various arrays.

  Args:
    Z: numpy.ndarray of shape (3, 4)

  Returns:
    A: jnp.ndarray — shape (20, 21) of ones
    B: jnp.ndarray — values equal to Z
    C: jnp.ndarray — same shape as A, values ~ Uniform(0, 1)
    D: jnp.ndarray — 1D array of even numbers from 4 to 40 inclusive
  """
  A = jnp.ones((20, 21))
  B = jnp.array(Z)
  key = jrandom.PRNGKey(0)
  C = jrandom.uniform(key, shape=A.shape)
  D = jnp.arange(4, 41, step=2)
  return A, B, C, D

Z = np.random.rand(3, 4)
A, B, C, D = array_creation(Z)
checkExercise1(A, B, C, D)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Creating_Tensors_Exercise")

## Section 2.2: Operations in JAX

**Array-Array operations**

JAX overloads all standard Python operators just like PyTorch and NumPy.

In [ ]:
# @title Video 4: Tensor Operators
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Tensors_Operators_Video")

In [ ]:
a = jnp.ones((5, 3))
key = jrandom.PRNGKey(1)
b = jrandom.uniform(key, shape=(5, 3))

# Element-wise addition and multiplication
c = a + b
d = a * b

print(c)
print(d)

In [ ]:
x = jnp.array([1, 2, 4, 8])
y = jnp.array([1, 2, 3, 4])
x + y, x - y, x * y, x / y, x**y

**Array methods — sum, mean:**

In [ ]:
key = jrandom.PRNGKey(0)
x = jrandom.uniform(key, shape=(3, 3))
print(x)
print()
print(f"Sum of every element: {x.sum()}")
print(f"Sum of columns (axis=0): {x.sum(axis=0)}")
print(f"Sum of rows (axis=1): {x.sum(axis=1)}")
print()
print(f"Mean of all elements: {x.mean()}")
print(f"Mean of columns: {x.mean(axis=0)}")
print(f"Mean of rows: {x.mean(axis=1)}")

**Matrix operations**

The `@` operator works for matrix multiplication, just as in PyTorch and NumPy.

### Coding Exercise 2.2: Simple array operations

In [ ]:
def simple_operations(a1: jnp.ndarray, a2: jnp.ndarray, a3: jnp.ndarray):
  """
  Multiplication of a1 with a2, then add a3.

  Args:
    a1, a2, a3: jnp.ndarray of shape (2, 2)

  Returns:
    answer: jnp.ndarray of shape (2, 2)
  """
  #################################################
  ## TODO for students
  raise NotImplementedError("Student exercise")
  #################################################
  answer = ...
  return answer

In [ ]:
# to_remove solution
def simple_operations(a1: jnp.ndarray, a2: jnp.ndarray, a3: jnp.ndarray):
  """
  Multiplication of a1 with a2, then add a3.
  """
  answer = a1 @ a2 + a3
  return answer

key = jrandom.PRNGKey(0)
key, sk1 = jrandom.split(key)
key, sk2 = jrandom.split(key)
key, sk3 = jrandom.split(key)
a1 = jrandom.randint(sk1, (2,2), 0, 5).astype(float)
a2 = jrandom.randint(sk2, (2,2), 0, 5).astype(float)
a3 = jrandom.randint(sk3, (2,2), 0, 5).astype(float)
print(simple_operations(a1, a2, a3))

In [ ]:
def dot_product(b1: jnp.ndarray, b2: jnp.ndarray):
  """
  Compute the dot product of two 1D arrays.

  Args:
    b1, b2: jnp.ndarray of shape (3,)

  Returns:
    product: scalar jnp.ndarray
  """
  #################################################
  ## TODO for students
  raise NotImplementedError("Student exercise")
  #################################################
  product = ...
  return product

In [ ]:
# to_remove solution
def dot_product(b1: jnp.ndarray, b2: jnp.ndarray):
  product = jnp.dot(b1, b2)
  return product

b1 = jnp.array([1., 2., 3.])
b2 = jnp.array([4., 5., 6.])
print(dot_product(b1, b2))  # Expected: 32.0

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Simple_Tensor_Operations_Exercise")

## Section 2.3: Manipulating Arrays in JAX

In [ ]:
# @title Video 5: Tensor Indexing
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Manipulating_Tensors_Video")

**Indexing** — identical to NumPy:

In [ ]:
x = jnp.arange(0, 10)
print(x)
print(x[-1])
print(x[1:3])
print(x[:-2])

> **Note:** JAX supports most NumPy indexing, but **in-place indexed assignment
> (`x[0] = 5`) is not allowed** — JAX arrays are immutable.
> Use `x = x.at[0].set(5)` instead.

In [ ]:
# JAX in-place-style assignment (functional, returns a new array)
x = jnp.arange(10, dtype=float)
x = x.at[0].set(99.0)   # instead of x[0] = 99.0
print(x)

In [ ]:
# 5D array indexing
key = jrandom.PRNGKey(0)
x = jrandom.uniform(key, shape=(1, 2, 3, 4, 5))
print(f"shape of x[0]: {x[0].shape}")
print(f"shape of x[0][0]: {x[0][0].shape}")
print(f"shape of x[0][0][0]: {x[0][0][0].shape}")

**Flatten and reshape:**

In [ ]:
z = jnp.arange(12).reshape(6, 2)
print(f"Original z:\n{z}")

z = z.flatten()
print(f"Flattened z:\n{z}")

z = z.reshape(3, 4)
print(f"Reshaped (3x4) z:\n{z}")

**Squeezing arrays:**

In [ ]:
key = jrandom.PRNGKey(0)
x = jrandom.normal(key, shape=(1, 10))
print(x.shape)
print(f"x[0]: {x[0]}")   # Returns a row, not a scalar!

# Remove the singleton dimension
x = jnp.squeeze(x, axis=0)   # equivalent to torch.squeeze(0)
print(x.shape)
print(f"x[0]: {x[0]}")   # Now returns the first element

In [ ]:
# Adding singleton dimensions: jnp.expand_dims  (equivalent to torch.unsqueeze)
key = jrandom.PRNGKey(0)
y = jrandom.normal(key, shape=(5, 5))
print(f"Shape of y: {y.shape}")

y = jnp.expand_dims(y, axis=1)   # torch.unsqueeze(1)
print(f"Shape of y after expand_dims: {y.shape}")

**Permutation (reordering axes):**

In [ ]:
# x has dimensions [color, image_height, image_width]
key = jrandom.PRNGKey(0)
x = jrandom.uniform(key, shape=(3, 48, 64))

# Reorder to [image_height, image_width, color]
x = jnp.transpose(x, (1, 2, 0))   # equivalent to torch.permute(1, 2, 0)
print(x.shape)

**Concatenation:**

In [ ]:
x = jnp.arange(12, dtype=jnp.float32).reshape((3, 4))
y = jnp.array([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

cat_rows = jnp.concatenate([x, y], axis=0)   # torch.cat(..., dim=0)
cat_cols = jnp.concatenate([x, y], axis=1)

print(f"Concatenated by rows: shape {list(cat_rows.shape)}\n{cat_rows}")
print(f"\nConcatenated by columns: shape {list(cat_cols.shape)}\n{cat_cols}")

**Conversion to/from NumPy:**

In [ ]:
key = jrandom.PRNGKey(0)
x = jrandom.normal(key, shape=(5,))
print(f"x: {x}  |  x type: {type(x)}")

y = np.array(x)                # equivalent to x.numpy()
print(f"y: {y}  |  y type: {type(y)}")

z = jnp.array(y)              # equivalent to torch.tensor(y)
print(f"z: {z}  |  z type: {type(z)}")

In [ ]:
a = jnp.array([3.5])
print(a, a.item(), float(a), int(a))

### Coding Exercise 2.3: Manipulating Arrays

In [ ]:
def functionA(my_array1, my_array2):
  """
  Returns the column sum of my_array1 multiplied by the sum of all elements of my_array2.

  Args:
    my_array1: jnp.ndarray (2D)
    my_array2: jnp.ndarray (2D)

  Returns:
    output: jnp.ndarray
  """
  ################################################
  ## TODO for students
  raise NotImplementedError("Student exercise")
  ################################################
  output = ...
  return output


def functionB(my_array):
  """
  Returns the transpose of my_array with a singleton dimension inserted at axis=0.

  Args:
    my_array: jnp.ndarray (2D)

  Returns:
    output: jnp.ndarray
  """
  ################################################
  ## TODO for students
  raise NotImplementedError("Student exercise")
  ################################################
  output = ...
  return output

In [ ]:
# to_remove solution
def functionA(my_array1, my_array2):
  output = my_array1.sum(axis=0) * my_array2.sum()
  return output

def functionB(my_array):
  output = jnp.expand_dims(my_array.T, axis=0)
  return output

key = jrandom.PRNGKey(0)
key, sk1 = jrandom.split(key)
key, sk2 = jrandom.split(key)
a1 = jrandom.randint(sk1, (3, 2), 0, 6).astype(float)
a2 = jrandom.randint(sk2, (2, 3), 0, 6).astype(float)
print("functionA:", functionA(a1, a2))
print("functionB:", functionB(a1))

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Manipulating_Tensors_Exercise")

## Section 2.4: Devices in JAX

In [ ]:
# @title Video 6: GPU vs CPU
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_GPU_vs_CPU_Video")

**JAX handles devices transparently.**

Unlike PyTorch, you do not manually call `.to("cuda")` on every tensor.
JAX automatically places computations on the best available device.
You can query and manually control placement with `jax.device_put`.

In [ ]:
# Check available devices
print(jax.devices())

# Check default backend
print(jax.default_backend())

In [ ]:
# Create an array — JAX places it on the default device automatically
key = jrandom.PRNGKey(0)
x = jrandom.normal(key, shape=(10,))
print(x)
print(f"Device: {x.devices()}")

In [ ]:
def set_device():
  """
  Return the best available JAX device.

  Returns:
    device: jax.Device
  """
  gpus = jax.devices("gpu") if jax.default_backend() == "gpu" else []
  if gpus:
    print(f"GPU available: {gpus[0]}")
    return gpus[0]
  else:
    print("GPU not available, using CPU.")
    return jax.devices("cpu")[0]

DEVICE = set_device()

In [ ]:
# Move an array to a specific device
key = jrandom.PRNGKey(0)
x = jrandom.normal(key, shape=(2, 2))
x_on_device = jax.device_put(x, DEVICE)  # equivalent to x.to(device)
print(f"x device: {x_on_device.devices()}")

# Move back to CPU
x_cpu = jax.device_put(x_on_device, jax.devices("cpu")[0])
print(f"x_cpu device: {x_cpu.devices()}")

### Coding Exercise 2.4: How much faster is GPU?

In [ ]:
dim = 10000
iterations = 1

In [ ]:
def simpleFun(dim, device):
  """
  Perform matrix operations on a given device and time them.

  Args:
    dim: int
    device: jax.Device

  Returns:
    Nothing.
  """
  ################################################
  ## TODO for students: fill in the missing code
  raise NotImplementedError("Student exercise")
  ################################################
  key = jrandom.PRNGKey(0)
  # Create two dim x dim uniform arrays on `device`
  x = ...
  y = ...
  # Scalar-filled array
  z = ...
  # Element-wise multiply, then matrix multiply
  a = ...
  b = ...

In [ ]:
# to_remove solution
def simpleFun(dim, device):
  key = jrandom.PRNGKey(0)
  key, sk1 = jrandom.split(key)
  key, sk2 = jrandom.split(key)
  x = jax.device_put(jrandom.uniform(sk1, (dim, dim)), device)
  y = jax.device_put(jrandom.uniform(sk2, (dim, dim)), device)
  z = jax.device_put(jnp.full((dim, dim), 2.0), device)
  a = x * y
  b = x @ y
  b.block_until_ready()  # Wait for async computation to finish before timing

for device in jax.devices():
  start = time.time()
  for _ in range(iterations):
    simpleFun(dim, device)
  elapsed = time.time() - start
  print(f"Time on {device}: {elapsed:.4f}s for {iterations} iteration(s)")

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_How_Much_Faster_Are_GPUs_Exercise")

> **Note:** JAX uses **asynchronous dispatch** — operations are submitted to the device
> and Python continues immediately. Always call `.block_until_ready()` before timing to get accurate results.

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_GPUs_Discussion")

## Section 2.5: Datasets and Dataloaders

In [ ]:
# @title Video 7: Getting Data
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Getting_Data_Video")

When training models you will be working with large amounts of data.
We keep **torchvision DataLoaders** for data management — they are convenient and well-tested.
The only change: we convert each batch to a JAX array using `jnp.array(batch.numpy())`.

In [ ]:
# Import dataset and dataloaders related packages
from torchvision import datasets
from torchvision.transforms import ToTensor, Compose, Grayscale
from torch.utils.data import DataLoader

In [ ]:
# Download and load the images from the CIFAR10 dataset
cifar10_data = datasets.CIFAR10(
    root="data",
    download=True,
    transform=ToTensor()
)
print(f"Number of samples: {len(cifar10_data)}")
print(f"Class names: {cifar10_data.classes}")

In [ ]:
# Choose a random sample
random.seed(2021)
image_torch, label = cifar10_data[random.randint(0, len(cifar10_data))]

# Convert to JAX array
image = jnp.array(image_torch.numpy())
print(f"Label: {cifar10_data.classes[label]}")
print(f"Image shape: {image.shape}")  # (C, H, W)

### Coding Exercise 2.5: Display an image from the dataset

In [ ]:
# TODO: Fix this code — images are (C, H, W) but plt.imshow expects (H, W, C)
# plt.imshow(np.array(image))   # Uncomment to see the error

# TODO: Reorder the axes using jnp.transpose and display the image
# plt.imshow(jnp.transpose(image, ...))
# plt.show()

In [ ]:
# to_remove solution
plt.imshow(jnp.transpose(image, (1, 2, 0)))   # (C,H,W) -> (H,W,C)
plt.show()

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Display_an_Image_Exercise")

In [ ]:
# @title Video 8: Train and Test
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Train_and_Test_Video")

In [ ]:
# Load training and test datasets
training_data = datasets.CIFAR10(root="data", train=True, download=True, transform=ToTensor())
test_data = datasets.CIFAR10(root="data", train=False, download=True, transform=ToTensor())

In [ ]:
# Create DataLoaders
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

In [ ]:
# Load a batch and convert to JAX arrays
batch_images_torch, batch_labels_torch = next(iter(train_dataloader))
batch_images = jnp.array(batch_images_torch.numpy())
batch_labels = jnp.array(batch_labels_torch.numpy())

print("Batch images shape:", batch_images.shape)
plt.imshow(jnp.transpose(batch_images[0], (1, 2, 0)))
plt.show()

### Coding Exercise 2.6: Load CIFAR10 as grayscale images

In [ ]:
def my_data_load():
  """
  Load CIFAR10 as grayscale images and display one.

  Returns:
    data: torchvision Dataset
  """
  ###############################################
  ## TODO for students: load CIFAR10 with grayscale transform
  raise NotImplementedError("Student exercise")
  ###############################################
  data = datasets.CIFAR10(root="data", download=True, transform=...)
  image_torch, label = data[random.randint(0, len(data))]
  image = jnp.array(image_torch.numpy())
  plt.imshow(jnp.squeeze(image), cmap="gray")
  plt.show()
  return data

In [ ]:
# to_remove solution
def my_data_load():
  data = datasets.CIFAR10(root="data", download=True,
                          transform=Compose([ToTensor(), Grayscale()]))
  random.seed(2021)
  image_torch, label = data[random.randint(0, len(data))]
  image = jnp.array(image_torch.numpy())
  plt.imshow(jnp.squeeze(image), cmap="gray")
  plt.show()
  return data

grayscale_data = my_data_load()

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Load_CIFAR10_Exercise")

---
# Section 3: Neural Networks with Flax

*Time estimate: ~1 hour 30 mins*

Now it is time to create your first neural network using **Flax**.
Flax is a neural network library built on top of JAX.

Key differences from PyTorch `nn.Module`:
- Parameters are **not stored inside the model** — they live in a separate `params` dict
- Forward pass is done with `model.apply(params, x)` not `model(x)`
- Use `@nn.compact` to define layers inline in `__call__`

In [ ]:
# @title Video 10: CSV Files
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_CSV_files_Video")

## Section 3.1: Data Loading

In [ ]:
# @title Generate sample data
from sklearn.datasets import make_blobs

# Generate 2D classification dataset
X_np, y_np = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

# Save to CSV
import pandas as pd
df = pd.DataFrame({"x1": X_np[:, 0], "x2": X_np[:, 1], "label": y_np})
df.to_csv("sample_data.csv", index=False)
print("Sample data saved.")

In [ ]:
# Load the data from CSV
data = pd.read_csv("sample_data.csv")
print(data.head())
print(f"Shape: {data.shape}")

In [ ]:
# Convert to JAX arrays
DEVICE = set_device()

X = jnp.array(data[["x1", "x2"]].values, dtype=jnp.float32)
y = jnp.array(data["label"].values, dtype=jnp.int32)

print(f"X shape: {X.shape}, y shape: {y.shape}")

# Plot
fig, ax = plt.subplots()
scatter = ax.scatter(np.array(X[:, 0]), np.array(X[:, 1]),
                     c=np.array(y), cmap="tab10", alpha=0.7)
plt.colorbar(scatter)
plt.title("Sample Data")
plt.show()

## Section 3.2: Create a Simple Neural Network

In [ ]:
# @title Video 11: Generating the Neural Network
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Generating_Neural_Network_Video")

We want a simple 3-layer network:
- Input layer: 2 features
- Hidden layer: 16 units, ReLU activation
- Output layer: 3 units (one per class)

In Flax, we define `__call__` with `@nn.compact` to declare layers inline.

In [ ]:
class NaiveNet(nn.Module):
  """A simple 3-layer MLP classifier."""

  @nn.compact
  def __call__(self, x):
    x = nn.Dense(16)(x)
    x = nn.relu(x)
    x = nn.Dense(3)(x)
    return x  # raw logits (no softmax — cross-entropy loss handles that)

# Initialize the model
model = NaiveNet()

# Create a dummy input to initialize parameters
key = jrandom.PRNGKey(0)
dummy_input = jnp.ones((1, 2))  # batch of 1, 2 features
params = model.init(key, dummy_input)

print("Model parameter shapes:")
for layer, weights in params["params"].items():
  for name, val in weights.items():
    print(f"  {layer}/{name}: {val.shape}")

### Coding Exercise 3.2: Classify some samples

In [ ]:
## TODO: Pass the first 5 samples through the model and print the logits
# X_samples = ...
# logits = ...
# print("Sample input:", X_samples)
# print("Logits:", logits)
# print("Predicted classes:", jnp.argmax(logits, axis=-1))

In [ ]:
# to_remove solution
X_samples = X[0:5]
logits = model.apply(params, X_samples)
print("Sample input:", X_samples)
print("Logits:", logits)
print("Predicted classes:", jnp.argmax(logits, axis=-1))

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Classify_some_examples_Exercise")

## Section 3.3: Train Your Neural Network

In [ ]:
# @title Video 12: Train the Network
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Train_the_Network_Video")

Training in JAX/Flax/Optax follows a functional pattern:

1. Define a **loss function** that takes `params` and returns a scalar
2. Use `jax.value_and_grad(loss_fn)(params)` to get loss + gradients
3. Use an **Optax optimizer** to compute parameter updates
4. Apply updates with `optax.apply_updates(params, updates)`
5. Wrap the whole step in `@jax.jit` for speed

> **JIT compilation** (`jax.jit`): The first time a JIT-compiled function runs,
> JAX traces it and compiles it to optimized XLA code for your hardware.
> Subsequent calls skip the Python overhead and run the compiled version directly.
> This can make training loops **10–100x faster**.

In [ ]:
# @title Helper function to plot decision boundary
def plot_decision_boundary(model, params, X, y, ax=None):
  x_min, x_max = float(X[:, 0].min()) - 1, float(X[:, 0].max()) + 1
  y_min, y_max = float(X[:, 1].min()) - 1, float(X[:, 1].max()) + 1
  xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                        np.linspace(y_min, y_max, 200))
  grid = jnp.array(np.c_[xx.ravel(), yy.ravel()], dtype=jnp.float32)
  logits = model.apply(params, grid)
  Z = np.array(jnp.argmax(logits, axis=-1)).reshape(xx.shape)
  if ax is None:
    fig, ax = plt.subplots()
  ax.contourf(xx, yy, Z, alpha=0.4, cmap="tab10")
  ax.scatter(np.array(X[:, 0]), np.array(X[:, 1]),
             c=np.array(y), cmap="tab10", edgecolors="k", s=20)
  return ax

In [ ]:
def train(X, y, n_epochs=200, lr=0.01, seed=42):
  """
  Train NaiveNet on dataset (X, y).

  Args:
    X: jnp.ndarray of shape (N, 2)
    y: jnp.ndarray of shape (N,) with integer class labels
    n_epochs: int
    lr: float
    seed: int

  Returns:
    params: trained parameters
    losses: list of per-epoch losses
  """
  key = jrandom.PRNGKey(seed)
  model = NaiveNet()
  params = model.init(key, X[:1])

  optimizer = optax.sgd(lr)
  opt_state = optimizer.init(params)

  @jax.jit
  def train_step(params, opt_state, x_batch, y_batch):
    def loss_fn(params):
      logits = model.apply(params, x_batch)
      loss = optax.softmax_cross_entropy_with_integer_labels(logits, y_batch).mean()
      return loss

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

  losses = []
  for epoch in range(n_epochs):
    params, opt_state, loss = train_step(params, opt_state, X, y)
    losses.append(float(loss))

  return params, losses

params, losses = train(X, y, n_epochs=200, lr=0.05)
print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
plt.plot(np.linspace(1, len(losses), len(losses)), losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

ax = plot_decision_boundary(model, params, X, y)
ax.set_title("Decision Boundary after Training")
plt.show()

---
# Section 3.4: JIT Compilation — What Is It?

**`jax.jit`** stands for **Just-In-Time compilation**.

### How it works

When you call a plain Python/JAX function, Python runs line-by-line:
each `jnp.dot`, `jnp.relu`, etc. is dispatched to the device one at a time.
This has significant overhead for deep networks.

When you wrap a function with `@jax.jit`:
1. **First call:** JAX *traces* the function — it runs through the code with abstract values
   to build a computation graph, then compiles that graph to optimized **XLA** machine code.
2. **Subsequent calls:** The compiled binary runs directly, bypassing Python entirely.

### Analogy

Think of it like a recipe vs. a production line:
- **Without JIT:** You read the recipe step-by-step every time you cook.
- **With JIT:** The first time, you set up a fully automated production line. After that, you just press "go".

### Constraints

JIT-compiled functions must be **pure** — no Python side effects:
- No in-place mutation (use `x.at[i].set(v)`)
- No Python control flow that depends on array *values* (use `jax.lax.cond` instead of `if arr > 0`)
- No global state

### Example

In [ ]:
import time

def matmul_slow(x, y):
  return x @ y

@jax.jit
def matmul_fast(x, y):
  return x @ y

key = jrandom.PRNGKey(0)
key, sk1 = jrandom.split(key)
key, sk2 = jrandom.split(key)
x = jrandom.normal(sk1, (2000, 2000))
y = jrandom.normal(sk2, (2000, 2000))

# Warm up JIT
_ = matmul_fast(x, y).block_until_ready()

# Time without JIT
t0 = time.time()
for _ in range(10):
  matmul_slow(x, y).block_until_ready()
print(f"Without JIT: {time.time() - t0:.3f}s")

# Time with JIT
t0 = time.time()
for _ in range(10):
  matmul_fast(x, y).block_until_ready()
print(f"With JIT:    {time.time() - t0:.3f}s")

### Coding Exercise 3.3: Tweak your network

In [ ]:
# to_remove explanation
"""
Try changing:
1. Number of epochs — more epochs = lower training loss (up to a point)
2. Learning rate — too high: unstable, too low: slow convergence
3. Hidden layer size — more units = more capacity
4. Optimizer — try optax.adam(lr) instead of optax.sgd(lr)
""";

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Tweak_your_Network_Exercise")

---
# Section 4: Ethics and Course Info

In [ ]:
# @title Video 15: Ethics
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Ethics_Video")

In [ ]:
# @title Video 16: Be a group
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Be_a_group_Video")

In [ ]:
# @title Video 17: Syllabus
from ipywidgets import widgets
from IPython.display import YouTubeVideo, IFrame, display

class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)

def display_videos(video_ids, b=widgets.ToggleButtons()):
  def on_value_change(change):
    pbar.value = 0
  b.observe(on_value_change, names='value')
  output = widgets.Output()
  dct = dict(video_ids)
  with output:
    for val in b.options:
      if dct[val] == 'Youtube':
        iframe = YouTubeVideo(id=str(dct[val]), width=854, height=480, allow="autoplay")
      else:
        iframe = PlayVideo(id=str(dct[val]), source=dct[val], width=854, height=480, allow="autoplay")
      display(iframe)
  display(b, output)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Syllabus_Video")

---
# Appendix

## Official JAX/Flax resources:

### JAX
- [JAX documentation](https://jax.readthedocs.io/)
- [JAX 101 tutorials](https://jax.readthedocs.io/en/latest/jax-101/index.html)

### Flax
- [Flax documentation](https://flax.readthedocs.io/)
- [Flax examples](https://github.com/google/flax/tree/main/examples)

### Optax
- [Optax documentation](https://optax.readthedocs.io/)